<a href="https://colab.research.google.com/github/12halima/Transport_Recommander/blob/main/data/GTFS_CLEAN/GTFS_edges_data_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
!pip install pyspark


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.3/455.3 MB 926.3 kB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.0/203.0 kB 19.4 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-4.1.0-py2.py3-none-any.whl size=455986285 sha256=f07e8429162e47b5b0de641229363a6791bdb0c7b91292641b4a4132d2d0ccb8
  Stored in directory: /root/.cache/pip/wheels/6b/9b/7c/2bea6ee44c4721d1af223365b8ab673a2db3ed4b759c12439d
Successfully built pyspark


In [5]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("GTFS-FULL-DATASET-TESTS")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark


In [6]:
BASE_PATH = "/content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_2"

df = (
    spark.read
    .option("recursiveFileLookup", "true")
    .parquet(BASE_PATH)
)

print("FULL dataset chargé")
print("Nombre de lignes :", df.count())
df.printSchema()


FULL dataset chargé
Nombre de lignes : 3210579
root
 |-- city: string (nullable = true)
 |-- trip_id: string (nullable = true)
 |-- route_short_name: string (nullable = true)
 |-- route_long_name: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- route_type: string (nullable = true)
 |-- from_stop_id: string (nullable = true)
 |-- to_stop_id: string (nullable = true)
 |-- from_lat: double (nullable = true)
 |-- from_lon: double (nullable = true)
 |-- to_lat: double (nullable = true)
 |-- to_lon: double (nullable = true)
 |-- from_arrival_time: string (nullable = true)
 |-- from_departure_time: string (nullable = true)
 |-- to_arrival_time: string (nullable = true)
 |-- to_departure_time: string (nullable = true)
 |-- travel_time_sec: long (nullable = true)
 |-- travel_time_departure_sec: long (nullable = true)
 |-- delta_lat: double (nullable = true)
 |-- delta_lon: double (nullable = true)



In [7]:
# =========================================================
# 3️⃣ Test SCHÉMA : présence des colonnes attendues
# =========================================================

expected_columns = [
    "city",
    "trip_id",
    "route_short_name",
    "route_long_name",
    "route_id",
    "route_type",
    "from_stop_id",
    "to_stop_id",
    "from_lat",
    "from_lon",
    "to_lat",
    "to_lon",
    "from_arrival_time",
    "from_departure_time",
    "to_arrival_time",
    "to_departure_time",
    "travel_time_sec",
    "travel_time_departure_sec",
    "delta_lat",
    "delta_lon"
]

# Colonnes manquantes
missing_columns = [c for c in expected_columns if c not in df.columns]

# Colonnes en trop (optionnel mais utile)
extra_columns = [c for c in df.columns if c not in expected_columns]

# Assertions
assert not missing_columns, f"❌ Colonnes manquantes ({len(missing_columns)}) : {missing_columns}"

print("✅ TEST SCHÉMA : OK")
print(f"   ➜ Colonnes attendues : {len(expected_columns)}")
print(f"   ➜ Colonnes présentes : {len(df.columns)}")
print(f"   ➜ Colonnes manquantes : {len(missing_columns)}")
print(f"   ➜ Colonnes en trop : {len(extra_columns)}")

if extra_columns:
    print("ℹ️ Colonnes supplémentaires détectées :", extra_columns)


✅ TEST SCHÉMA : OK
   ➜ Colonnes attendues : 20
   ➜ Colonnes présentes : 20
   ➜ Colonnes manquantes : 0
   ➜ Colonnes en trop : 0


In [9]:
from pyspark.sql.functions import col

df = df.withColumn(
    "route_type",
    col("route_type").cast("bigint")
)


In [10]:
from pyspark.sql.functions import col, to_timestamp, when

# =========================================================
# 4️⃣ TEST SCHÉMA — TYPES
# =========================================================

expected_types = {
    "city": "string",
    "trip_id": "string",
    "route_id": "string",
    "route_short_name": "string",
    "route_long_name": "string",
    "route_type": "bigint",
    "from_stop_id": "string",
    "to_stop_id": "string",
    "from_lat": "double",
    "from_lon": "double",
    "to_lat": "double",
    "to_lon": "double",
    "travel_time_sec": "bigint",
    "travel_time_departure_sec": "bigint",
    "delta_lat": "double",
    "delta_lon": "double"
}

dtypes = dict(df.dtypes)

wrong_types = [
    (col_name, dtypes.get(col_name), expected)
    for col_name, expected in expected_types.items()
    if dtypes.get(col_name) != expected
]

assert not wrong_types, f"❌ Types incorrects ({len(wrong_types)}) : {wrong_types}"

print("✅ TEST TYPES : OK")


# =========================================================
# 5️⃣ TEST FORMAT TEMPS — HH:mm:ss
# =========================================================

time_columns = [
    "from_arrival_time",
    "from_departure_time",
    "to_arrival_time",
    "to_departure_time"
]

# Conversion sécurisée vers timestamp
df_time = df
for c in time_columns:
    df_time = df_time.withColumn(
        f"{c}_ts",
        to_timestamp(col(c), "HH:mm:ss")
    )

# Lignes avec format invalide
invalid_time_rows = df_time.filter(
    sum(col(f"{c}_ts").isNull().cast("int") for c in time_columns) > 0
).count()

assert invalid_time_rows == 0, f"❌ {invalid_time_rows} lignes avec format temps invalide"

print("✅ TEST FORMAT TEMPS (HH:mm:ss) : OK")


# =========================================================
# 6️⃣ TEST COHÉRENCE TEMPORELLE
# departure <= arrival
# =========================================================

time_consistency_errors = df_time.filter(
    (col("from_departure_time_ts") > col("from_arrival_time_ts")) |
    (col("to_departure_time_ts") > col("to_arrival_time_ts"))
).count()

assert time_consistency_errors == 0, (
    f"❌ Incohérences temporelles détectées : {time_consistency_errors} lignes "
    "(departure > arrival)"
)

print("✅ TEST COHÉRENCE TEMPORELLE : OK")


# =========================================================
# 🎉 RÉSUMÉ
# =========================================================

print("\n🎯 TOUS LES TESTS SCHÉMA & TEMPS SONT VALIDÉS AVEC SUCCÈS")


✅ TEST TYPES : OK
✅ TEST FORMAT TEMPS (HH:mm:ss) : OK
✅ TEST COHÉRENCE TEMPORELLE : OK

🎯 TOUS LES TESTS SCHÉMA & TEMPS SONT VALIDÉS AVEC SUCCÈS


In [11]:
# ✅ 5️⃣ Clés critiques non nulles
null_keys = df.filter(
    df.trip_id.isNull() | df.from_stop_id.isNull() | df.to_stop_id.isNull()
).count()

# Vérification
assert null_keys == 0, f"Lignes avec clés nulles ({null_keys})"

print(f"TEST QUALITÉ (clés non nulles) : OK (lignes avec clés nulles = {null_keys})")


TEST QUALITÉ (clés non nulles) : OK (lignes avec clés nulles = 0)


In [12]:
# 🚦 6️⃣ Séparation self-loops / normal edges
self_loops = df.filter(df.from_stop_id == df.to_stop_id)
normal_edges = df.filter(df.from_stop_id != df.to_stop_id)

# Compter
self_loops_count = self_loops.count()
normal_edges_count = normal_edges.count()

print(f"Self-loops : {self_loops_count}")
print(f"Normal edges : {normal_edges_count}")
print(f"Total edges : {self_loops_count + normal_edges_count}")


Self-loops : 290934
Normal edges : 2919645
Total edges : 3210579


In [ ]:
# Afficher un échantillon des self-loops
print("Exemple de self-loops :")
self_loops.show(10, truncate=False)

Exemple de self-loops :
+-----------+------------------------------------+------------------------------------+------------------------------------+------------------------------------+----------+----------+----------+----------+---------------+---------+---------+
|city       |trip_id                             |route_id                            |from_stop_id                        |to_stop_id                          |from_lat  |from_lon  |to_lat    |to_lon    |travel_time_sec|delta_lat|delta_lon|
+-----------+------------------------------------+------------------------------------+------------------------------------+------------------------------------+----------+----------+----------+----------+---------------+---------+---------+
|TranspRober|00016edc-e5e4-42e9-9078-81966d13233e|eb50d256-a74e-4008-ac22-4fb4ad7bf4e4|a0c48aed-25e2-4fe8-ad4c-d70f64598737|a0c48aed-25e2-4fe8-ad4c-d70f64598737|37.1541386|-3.5926258|37.1541386|-3.5926258|0              |0.0      |0.0      |
|TranspR

In [13]:
from pyspark.sql.functions import abs,when

# 🚦 7️⃣ Correction self-loops (travel_time_sec = 0, delta ≈ 0)

# Compter avant correction
valid_self_loops_before = self_loops.filter(
    (self_loops.travel_time_sec == 0) &
    (abs(self_loops.delta_lat) <= 1e-6) &
    (abs(self_loops.delta_lon) <= 1e-6)
).count()

print(f"Avant correction : {valid_self_loops_before} self-loops déjà corrects")



self_loops_clean = self_loops.withColumn(
    "travel_time_sec",
    when(col("travel_time_sec") != 0, 0).otherwise(col("travel_time_sec"))
).withColumn(
    "delta_lat",
    when(col("from_stop_id") == col("to_stop_id"), 0.0)
    .otherwise(col("delta_lat"))
).withColumn(
    "delta_lon",
    when(col("from_stop_id") == col("to_stop_id"), 0.0)
    .otherwise(col("delta_lon"))
)


Avant correction : 290859 self-loops déjà corrects


In [ ]:
from pyspark.sql.functions import when, col

# ✅ 8️⃣ Correction edges normaux (travel_time_sec ≥ 0)

# Compter avant correction
invalid_normal_edges_before = normal_edges.filter(
    col("travel_time_sec") <= 0
).count()

print(f"Avant correction : {invalid_normal_edges_before} edges normaux avec travel_time_sec <= 0")








Avant correction : 35746 edges normaux avec travel_time_sec <= 0


In [ ]:
# 👉 Afficher un échantillon des edges invalides
print("Exemple d'edges normaux invalides :")
normal_edges.filter(col("travel_time_sec") <= 0).show(10, truncate=False) # affiche 10 lignes

Exemple d'edges normaux invalides :
+---------------------------------------------------------------------------------------+-------------------------+---------+------------+-----------+----------------+-----------------+----------------+-----------------+---------------+---------------------+---------------------+
|city                                                                                   |trip_id                  |route_id |from_stop_id|to_stop_id |from_lat        |from_lon         |to_lat          |to_lon           |travel_time_sec|delta_lat            |delta_lon            |
+---------------------------------------------------------------------------------------+-------------------------+---------+------------+-----------+----------------+-----------------+----------------+-----------------+---------------+---------------------+---------------------+
|Consorcio Regional de Transportes de Madrid CRTM Intercity Buses (Madrid Intercity Bus)|1318946_8__611___-9560652|8__611

In [14]:
# Correction
normal_edges_clean = normal_edges.withColumn(
    "travel_time_sec",
    when(col("travel_time_sec") <= 0, 120).otherwise(col("travel_time_sec"))
)

# Vérification après correction
invalid_normal_edges_after = normal_edges_clean.filter(
    col("travel_time_sec") <= 0
).count()

print(f"Après correction : {invalid_normal_edges_after} edges normaux incorrects restants")


Après correction : 0 edges normaux incorrects restants


In [15]:
# ✅ 9️⃣ Reconstitution df_clean et suppression doublons

# Compter les doublons avant suppression
duplicates_before = (
    self_loops_clean.unionByName(normal_edges_clean)
    .groupBy("trip_id", "from_stop_id", "to_stop_id", "travel_time_sec")
    .count()
    .filter("count > 1")
    .count()
)

print(f"Doublons avant suppression : {duplicates_before}")

# Suppression des doublons
df_clean = self_loops_clean.unionByName(normal_edges_clean)
df_clean = df_clean.dropDuplicates(["trip_id","from_stop_id","to_stop_id","travel_time_sec"])

# Vérification après suppression
duplicates_after = (
    df_clean.groupBy("trip_id", "from_stop_id", "to_stop_id", "travel_time_sec")
    .count()
    .filter("count > 1")
    .count()
)

print(f"Doublons après suppression : {duplicates_after}")


Doublons avant suppression : 2397
Doublons après suppression : 0


In [16]:
# ✅ 10️⃣ Recalcul delta + forçage self-loops
df_clean = df_clean.withColumn(
    "delta_lat",
    when(col("from_stop_id") == col("to_stop_id"), 0.0)
    .otherwise(col("to_lat") - col("from_lat"))
).withColumn(
    "delta_lon",
    when(col("from_stop_id") == col("to_stop_id"), 0.0)
    .otherwise(col("to_lon") - col("from_lon"))
)


In [ ]:
# ✅ Vérification travel_time_sec > 0 pour edges normaux
time_errors = df_clean.filter(
    (col("from_stop_id") != col("to_stop_id")) &
    (col("travel_time_sec") <= 0)
).count()

# Vérification
assert time_errors == 0, f"Edges normaux avec travel_time_sec <= 0 : {time_errors}"

print(f"TEST MÉTIER (travel_time_sec > 0 pour edges normaux) : OK (edges incorrects = {time_errors})")


TEST MÉTIER (travel_time_sec > 0 pour edges normaux) : OK (edges incorrects = 0)


In [17]:
from pyspark.sql.functions import col

# ✅ 12️⃣ Coordonnées géographiques valides
geo_errors = df_clean.filter(
    (col("from_lat") < -90) | (col("from_lat") > 90) |
    (col("to_lat") < -90) | (col("to_lat") > 90) |
    (col("from_lon") < -180) | (col("from_lon") > 180) |
    (col("to_lon") < -180) | (col("to_lon") > 180)
).count()

# Vérification
assert geo_errors == 0, f"Coordonnées invalides ({geo_errors})"

print(f"TEST QUALITÉ (lat/lon) : OK (coordonnées invalides = {geo_errors})")


TEST QUALITÉ (lat/lon) : OK (coordonnées invalides = 0)


In [18]:
# =========================================================
# 13️⃣ Suppression des doublons EXACTS (toutes colonnes)
# =========================================================


df_clean = df_clean.dropDuplicates()

from pyspark import StorageLevel

edges_final = df_clean.persist(StorageLevel.MEMORY_AND_DISK)

edges_final.count()  # matérialisation sécurisée

print("✅ edges_final persisté (MEMORY + DISK)")




📈 Lignes après : 3207999


In [ ]:
# ✅ 14️⃣ Ratio self-loops < 20%
self_loops_count = df_clean.filter(col("from_stop_id") == col("to_stop_id")).count()
ratio = self_loops_count / df_clean.count()
assert ratio < 0.2
print(f"Ratio self-loops : {ratio:.2%}")


Ratio self-loops : 11.51%


In [ ]:
from pyspark.sql.functions import col, sum as spark_sum

# ✅ 15️⃣ Chaque trip a au moins un edge normal
bad_trips = (
    df_clean.groupBy("trip_id")
    .agg(spark_sum((col("from_stop_id") != col("to_stop_id")).cast("int")).alias("normal_edges"))
    .filter(col("normal_edges") == 0)
    .count()
)

# Vérification
assert bad_trips == 0, f"Trips sans edges normaux ({bad_trips})"

print(f"TEST TRIPS (au moins un edge normal) : OK (trips incorrects = {bad_trips})")


TEST TRIPS (au moins un edge normal) : OK (trips incorrects = 0)


In [ ]:
from pyspark.sql.functions import col, abs

# ✅ 16️⃣ Self-loops temps et delta
invalid_self_loop_time = df_clean.filter(
    (col("from_stop_id") == col("to_stop_id")) & (col("travel_time_sec") != 0)
).count()

invalid_self_loop_distance = df_clean.filter(
    (col("from_stop_id") == col("to_stop_id")) &
    ((abs(col("delta_lat")) > 1e-6) | (abs(col("delta_lon")) > 1e-6))
).count()

# Affichage des résultats avant vérification
print(f"Self-loops avec temps != 0 : {invalid_self_loop_time}")
print(f"Self-loops avec déplacement != 0 : {invalid_self_loop_distance}")

# Vérification
assert invalid_self_loop_time == 0, f"Self-loops avec temps != 0 ({invalid_self_loop_time})"
assert invalid_self_loop_distance == 0, f"Self-loops avec déplacement != 0 ({invalid_self_loop_distance})"

print(f"TEST SELF-LOOPS : OK (temps incorrects = {invalid_self_loop_time}, déplacements incorrects = {invalid_self_loop_distance})")


Self-loops avec temps != 0 : 0
Self-loops avec déplacement != 0 : 0
TEST SELF-LOOPS : OK (temps incorrects = 0, déplacements incorrects = 0)


In [20]:
from pyspark import StorageLevel

edges_final = df_clean.persist(StorageLevel.MEMORY_AND_DISK)


print("✅ edges_final persisté (MEMORY + DISK)")


ConnectionRefusedError: [Errno 111] Connection refused

In [ ]:
import os

OUTPUT_EDGES_BASE = "/content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_CLEAN_1"
os.makedirs(OUTPUT_EDGES_BASE, exist_ok=True)
cities_in_df = (
    edges_final
    .select("city")
    .distinct()
    .rdd
    .map(lambda r: r.city)
    .collect()
)

print("Nombre de villes à sauvegarder :", len(cities_in_df))
for city_name in cities_in_df:
    # Nom de dossier sûr
    safe_city_name = (
        city_name
        .replace("/", "_")
        .replace(" ", "_")
        .replace("'", "_")
    )

    # Filtrage STRICT du dataset validé
    city_edges = edges_final.filter(col("city") == city_name)

    output_path = os.path.join(OUTPUT_EDGES_BASE, safe_city_name)

    (
        city_edges
        .write
        .mode("overwrite")
        .parquet(output_path)
    )

    num_rows = city_edges.count()
    print(f"✅ {city_name} → {num_rows} edges sauvegardés")
total_saved = 0

for city_name in cities_in_df:
    safe_city_name = city_name.replace("/", "_").replace(" ", "_").replace("'", "_")
    path = os.path.join(OUTPUT_EDGES_BASE, safe_city_name)
    total_saved += spark.read.parquet(path).count()

assert total_saved == edges_final.count(), "❌ Incohérence lors de la sauvegarde"
print("✅ Vérification globale post-sauvegarde : OK")


Nombre de villes à sauvegarder : 47
✅ Lurraldebus - Hernani Urban (bus urbain d’Hernani) → 350 edges sauvegardés
✅ Oñati urbain (Oñatiko herribusa) → 862 edges sauvegardés
✅ Vectalia Movilidad (bus de la ville de Cáceres) → 274828 edges sauvegardés
✅ TRAM Alicante → 55189 edges sauvegardés
✅ AUCORSA (Autobuses de Córdoba S.A.) → 123295 edges sauvegardés
✅ Sopela Town Hall → 319 edges sauvegardés
✅ AISA (Bus Madrid-Aranda de Duero-Burgo de Osma) → 339 edges sauvegardés
✅ Lurraldebus Guipuzcoana (La Guipuzcoana) → 25357 edges sauvegardés
✅ Consorcio Regional de Transportes de Madrid CRTM Madrid City Bus (Autobus urbano de Madrid) → 182545 edges sauvegardés
✅ Lurraldebus Zarautz → 2486 edges sauvegardés
✅ TranspRober → 276175 edges sauvegardés
✅ Transports Municipaux D’Egara (TMESA) Terrassa bus urbain → 118403 edges sauvegardés
✅ Junta de Extremadura (Bus du gouvernement régional d’Estrémadure) → 3975 edges sauvegardés
✅ Direxis TGO (Transportes Generales de Olesa) → 1547 edges sau

In [ ]:
# Nombre total de lignes dans edges_final
total_edges = edges_final.count()
print(f"Nombre total de lignes dans edges_final : {total_edges}")


Nombre total de lignes dans edges_final : 4586993
